In [172]:
# Célula 0 — Imports e configuração
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.cloud import bigquery

# === Paleta de Cores (semântica explícita: fluxo vs status) ===
COLORS = {
    # Cores de fluxo (identidade, não performance)
    "fluxo_pa":   "#6B46C1",   # Roxo — Produto Acabado
    "fluxo_tri":  "#EAB308",   # Amarelo — Triangulação

    # Cores de status (performance vs target)
    "status_ok":       "#16A34A",  # Verde — desvio ≤10d ou dentro do prazo
    "status_atencao":  "#F97316",  # Laranja — desvio 10–30d
    "status_critico":  "#DC2626",  # Vermelho — desvio >30d

    # Neutros
    "neutro_escuro":   "#374151",  # Cinza-escuro — texto, eixos
    "neutro_claro":    "#E5E7EB",  # Cinza-claro — células sem amostra
    "target_line":     "#9CA3AF",  # Cinza tracejado — linha de 120d
}
TEMPLATE = "plotly_white"
TARGET_LEAD_TIME = 120  # dias — padrão atual Insider

# === Configuração de Análise (parametrizável) ===
CONFIG = {
    # Janela temporal padrão (em meses corridos a partir de hoje)
    "janela_meses": 12,

    # Janela curta para comparação (tendência recente)
    "janela_meses_curta": 3,

    # Mínimo de OPs para incluir um agregado
    "min_ops_recomendacao": 5,
    "min_ops_grafico":      3,

    # Percentil usado para calcular o prazo recomendado por fornecedor×fluxo
    # 0.50 = mediana (agressivo) | 0.75 = padrão | 0.90 = conservador
    "percentil_recomendacao": 0.75,

    # Thresholds de status (em dias)
    "threshold_desvio_atencao": 10,
    "threshold_desvio_critico": 30,

    # Buckets de volume (faixas absolutas, conforme metodologia do relatório)
    "buckets_volume": [0, 200, 500, 1000, 2000, 5000, float("inf")],
    "labels_volume": ["<200", "200-499", "500-999", "1000-1999", "2000-4999", "5000+"],

    # Quantos fornecedores mostrar nos gráficos com ranking
    "top_n_fornecedores_grafico": 15,
    "top_n_fornecedores_heatmap": 20,
}

PROJECT_ID = "insider-data-lake"  # ajuste se necessário

try:
    client = bigquery.Client(project=PROJECT_ID)
    print(f"✅ Conectado ao BigQuery — projeto: {client.project}")
except Exception as e:
    print(f"❌ Falha na conexão com BigQuery: {e}")
    print("Execute 'gcloud auth application-default login' no terminal e reinicie o kernel.")
    raise


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Python(61255) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


✅ Conectado ao BigQuery — projeto: insider-data-lake


In [173]:
# Célula 1 — Base de capacidade (fornecedores, produtos, lead time teórico, mark-up)
SQL_CAPACITY = """
WITH
full_price AS (
    SELECT *
    FROM `insider-data-lake.integrated.muninn_products`
),
fabric_costs AS(
    SELECT 
    mfs.id AS fabric_sku_id,
    mfs.fabric_id,
    mfs.knitting_factory_id,
    mfs.sku AS fabric_sku,
    mfs.invoice_fabric_name AS factory_fabric_name,
    mfs.unit_price,
    mfs.minimum_volume_per_order,
    mfs.multiple_volume_per_order,
    mf.name AS fabric_name,
    mf.article_id,
    ma.name AS article_name,
    ma.unit AS article_unit,
    mkf.supplier_id,
    ms.alias AS knitting_factory_name,
    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id
    WHERE mfs.status IN ('available')
    ),
fabric_min_max_cost AS (
    SELECT
    fc.fabric_id,
    fc.fabric_name,
    MIN(fc.unit_price) AS min_fabric_cost,
    MAX(fc.unit_price) AS max_fabric_cost,
    COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,
    ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names,
    FROM fabric_costs AS fc
    GROUP BY fc.fabric_id,
    fc.fabric_name
),
article_sku AS (
SELECT
    mpsf.product_sku_id,
    mps.sku,
    mps.sku_name,
    mps.product_id,
    s.sku_state,
    s.gender,
    s.color,
    s.size,
    s.product_name,
    mpsf.fabric_id,
    mf.name AS fabric_name,
    mpsf.consumption,
    fc.min_fabric_cost AS min_fabric_unitary_cost,
    fc.max_fabric_cost AS max_fabric_unitary_cost,
    fc.min_fabric_cost * mpsf.consumption AS min_fabric_cost,
    fc.max_fabric_cost * mpsf.consumption AS max_fabric_cost,
    ma.unit AS article_unit,
    ma.name AS article_name,
    mf.article_id,
    fc.number_knitting_factories,
    fc.knitting_factories_names
FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id
),
sku_fabric_costs AS (
    SELECT 
        a_sku.sku, a_sku.sku_state, a_sku.product_id,
        SUM(a_sku.min_fabric_cost) AS min_fabric_cost,
        SUM(a_sku.max_fabric_cost) AS max_fabric_cost,
        STRING_AGG(DISTINCT article_name, ',' ORDER BY article_name) AS article_names,
    FROM article_sku AS a_sku
    GROUP BY a_sku.sku, a_sku.sku_state, a_sku.product_id
),
article_freq AS (
    SELECT product_id, article_names, COUNT(*) AS freq
    FROM sku_fabric_costs GROUP BY product_id, article_names
),
top_article AS (
    SELECT product_id,
        ARRAY_AGG(article_names ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS most_common_article_names
    FROM article_freq GROUP BY product_id
),
avg_fabric_cost_prod AS (
    SELECT fc.product_id,
        AVG(fc.max_fabric_cost) AS max_fabric_cost,
        AVG(fc.min_fabric_cost) AS min_fabric_cost,
        t.most_common_article_names AS article_names
    FROM sku_fabric_costs AS fc
    LEFT JOIN top_article AS t ON fc.product_id = t.product_id
    GROUP BY fc.product_id, t.most_common_article_names
),
costs AS (
    SELECT
        amp.product_id, p.product_name,
        amp.apparel_manufacturer_id, amp.is_finished_product,
        amp.manufacturer_cost as manufacture_cost,
        fc.min_fabric_cost, fc.max_fabric_cost, fc.article_names,
        CASE WHEN amp.is_finished_product = True THEN amp.manufacturer_cost
             ELSE amp.manufacturer_cost + fc.max_fabric_cost END AS manufacturing_cost,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
    LEFT JOIN avg_fabric_cost_prod AS fc ON fc.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    WHERE amp.status IN ('available','approved','incubation')
),
base_intermediaria AS (
    SELECT
        ampup.apparel_manufacturer_production_unit_id,
        am.supplier_id, amp.apparel_manufacturer_id,
        s.alias, s.city, s.state, s.created_at AS date_supplier_creation,
        p.product_id, p.product_name,
        amp.is_finished_product, amp.order_minimum_volume, amp.lead_time,
        ampu.apparel_manufacturer_cell_number,
        MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS max_capacity,
        ampup.weekly_maximum_productive_capacity,
        fp.full_price, amp.status AS status_cell,
        4*ampup.weekly_maximum_productive_capacity AS monthly_capacity,
        4*MAX(ampup.weekly_maximum_productive_capacity) OVER (
            PARTITION BY ampup.apparel_manufacturer_production_unit_id
        ) AS cell_max_monthly_capacity,
        COUNT(DISTINCT am.supplier_id) OVER (PARTITION BY p.product_id) AS num_suppliers_per_product,
    FROM `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units_products` AS ampup
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers_products` AS amp
        ON amp.id = ampup.apparel_manufacturer_product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON p.product_id = amp.product_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturers` AS am ON am.id = amp.apparel_manufacturer_id
    LEFT JOIN `insider-data-lake.integrated.muninn_apparel_manufacturer_production_units` AS ampu
        ON ampu.id = ampup.apparel_manufacturer_production_unit_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS s ON s.id = am.supplier_id
    LEFT JOIN full_price AS fp ON p.product_name = fp.product_name
    WHERE amp.status IN ('available','approved','incubation')
        AND ampup.weekly_maximum_productive_capacity > 0
),
cell_products AS (
    SELECT apparel_manufacturer_production_unit_id,
        COUNT(DISTINCT b.product_name) AS n_products_in_cell,
        STRING_AGG(DISTINCT b.product_name, ', ') AS products_in_cell
    FROM base_intermediaria AS b
    GROUP BY apparel_manufacturer_production_unit_id
),
sku_data AS (
    SELECT ps.sku, ps.product_sku_id, ps.sku_name, sku_d.sku_state, sku_d.product_name,
        sku_d.family, sku_d.category, psf.fabric_id, a.name AS article_name
    FROM `insider-data-lake.integrated.muninn_product_skus` AS ps
    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus_fabrics` AS psf ON ps.product_sku_id = psf.product_sku_id
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS f ON psf.fabric_id = f.id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS a ON f.article_id = a.id
    LEFT JOIN `insider-data-lake.integrated.skus` AS sku_d ON sku_d.sku = ps.sku
    ORDER BY ps.product_sku_id, psf.fabric_id
),
dpi AS (
    SELECT product_name, SUM(treated_generated_revenue) AS treated_generated_revenue
    FROM `insider-data-lake.sop_silver.demand_prediction_input`
    WHERE DATE(reference_date) >= DATE_SUB(DATE_TRUNC(CURRENT_DATE(), MONTH), INTERVAL 3 MONTH)
        AND DATE(reference_date) <  DATE_TRUNC(CURRENT_DATE(), MONTH)
        AND product_name IS NOT NULL
    GROUP BY product_name
),
revenue_totals AS (
    SELECT product_name, treated_generated_revenue,
        SUM(treated_generated_revenue) OVER () AS total_treated_generated_revenue
    FROM dpi
),
cum AS (
    SELECT product_name, treated_generated_revenue, total_treated_generated_revenue,
        SUM(treated_generated_revenue) OVER (
            ORDER BY treated_generated_revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_treated_generated_revenue
    FROM revenue_totals
),
abc_curve AS (
    SELECT product_name, treated_generated_revenue,
        SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) AS cum_share,
        CASE
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.8 THEN 'A'
            WHEN SAFE_DIVIDE(cum_treated_generated_revenue, total_treated_generated_revenue) <= 0.95 THEN 'B'
            ELSE 'C'
        END AS tag_abc
    FROM cum
),
freqs AS (
    SELECT product_name, sku_state, article_name, family, category, COUNT(*) AS freq
    FROM sku_data GROUP BY product_name, sku_state, article_name, family, category
),
product_data AS (
    SELECT f.product_name, p.product_id,
        CASE WHEN SUM(CASE WHEN f.sku_state = 'ativo_perene' THEN 1 ELSE 0 END) > 0
             THEN 'ativo_perene'
             ELSE ARRAY_AGG(f.sku_state ORDER BY freq DESC LIMIT 1)[OFFSET(0)] END AS product_state,
        ARRAY_TO_STRING(ARRAY_AGG(DISTINCT f.article_name), ', ') AS article_name,
        ARRAY_AGG(f.family ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS family,
        ARRAY_AGG(f.category ORDER BY freq DESC LIMIT 1)[OFFSET(0)] AS category,
        abc.tag_abc
    FROM freqs AS f
    LEFT JOIN `insider-data-lake.integrated.muninn_products` AS p ON f.product_name = p.product_name
    LEFT JOIN abc_curve AS abc ON abc.product_name = f.product_name
    GROUP BY f.product_name, p.product_id, abc.tag_abc
)
SELECT
    b.*,
    c.manufacturing_cost, c.article_names,
    SAFE_DIVIDE(b.full_price, c.manufacturing_cost) AS mark_up,
    MIN(c.manufacturing_cost) OVER (PARTITION BY b.product_id, b.is_finished_product) AS min_manufacturing_cost,
    cp.n_products_in_cell, cp.products_in_cell,
    pd.tag_abc, pd.product_state,
FROM base_intermediaria AS b
LEFT JOIN cell_products AS cp ON b.apparel_manufacturer_production_unit_id = cp.apparel_manufacturer_production_unit_id
LEFT JOIN costs AS c ON b.apparel_manufacturer_id = c.apparel_manufacturer_id
    AND b.product_id = c.product_id AND b.is_finished_product = c.is_finished_product
LEFT JOIN product_data AS pd ON b.product_id = pd.product_id
WHERE pd.product_state NOT IN ('desativado')
"""

print("⏳ Carregando base de capacidade...")
df_capacity = client.query(SQL_CAPACITY).to_dataframe()
print(f"✅ df_capacity: {len(df_capacity):,} linhas | {df_capacity.shape[1]} colunas")
df_capacity.head(3)

⏳ Carregando base de capacidade...


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ df_capacity: 229 linhas | 28 colunas


,apparel_manufacturer_production_unit_id,supplier_id,apparel_manufacturer_id,alias,city,state,date_supplier_creation,product_id,product_name,is_finished_product,...,cell_max_monthly_capacity,num_suppliers_per_product,manufacturing_cost,article_names,mark_up,min_manufacturing_cost,n_products_in_cell,products_in_cell,tag_abc,product_state
0,421,14,60,DALOP,São Paulo,SP,2025-02-06 13:31:01.894,139,Regata Intech Feminino,True,...,4000,2,31.420000000,Modal,4.423933800,31.420000000,2,"Regata Intech Feminino, Regata Intech Masculino",C,ativo_perene
1,16,4,25,ART LIVRE,São Paulo,SP,2025-02-06 13:31:01.894,139,Regata Intech Feminino,True,...,4000,2,33.380000000,Modal,4.164170162,31.420000000,2,"Regata Intech Masculino, Regata Intech Feminino",C,ativo_perene
2,34,7,37,BAE BRASIL,Londrina,PR,2025-02-06 13:31:01.894,196,Calcinha Safe Feminino,False,...,18000,2,20.980150000,"Bio Anti Cloro,Micromodal",4.051448631,20.980150000,3,"Calcinha Minimal Feminino, Calcinha Brief Femi...",C,ativo_perene


In [174]:
# Célula 2 — OPs com etapas de produção (lead time realizado + stamps por fase)
#
# Fonte: `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history`
# Grão da tabela: (op_code, product_sku, ingestion_date) — snapshot diário por SKU.
# current_production_stage já normalizado; não é necessário tratamento de status.
# stamp_created_production_order = MIN(ingestion_date) geral = primeira aparição da OP.
# Stamps de etapa = MIN(ingestion_date) por stage. supplier_relationship_status embutido.
SQL_OPS = """
WITH CTE_OPS AS (
  SELECT
    h.op_code,

    -- Detalhes da OP
    ANY_VALUE(h.current_production_stage)                   AS current_production_stage,
    ANY_VALUE(COALESCE(h.is_finished_product_order, false)) AS is_finished_product_order,
    STRING_AGG(DISTINCT h.product_name)                     AS product_names,
    STRING_AGG(DISTINCT h.product_color)                    AS product_colors,
    STRING_AGG(DISTINCT h.status_sku)                       AS sku_status,
    ANY_VALUE(h.supplier_name)                              AS supplier_name,
    ANY_VALUE(h.cycle_name)                                 AS cycle_name,
    ANY_VALUE(h.production_order_type)                      AS production_order_type,
    ANY_VALUE(h.supplier_relationship_status)               AS supplier_relationship_status,
    MAX(h.planned_quantity_op)                              AS planned_quantity_op,
    MAX(h.received_quantity_op)                             AS received_quantity_op,
    MAX(h.dt_planned_production_start)                      AS dt_planned_production_start,
    MAX(h.dt_planned_production_end)                        AS dt_planned_production_end,
    MAX(h.dt_planned_entry_warehouse)                       AS dt_planned_entry_warehouse,
    MAX(h.dt_reviewed_entry_warehouse)                      AS dt_reviewed_entry_warehouse,
    MAX(h.dt_largest_entry_warehouse)                       AS dt_largest_entry_warehouse,

    -- Stamps: primeiro ingestion_date em que cada estágio aparece para o op_code
    MIN(CAST(h.ingestion_date AS TIMESTAMP))
      AS stamp_created_production_order,
    MIN(CASE WHEN h.current_production_stage = 'order_request_validation'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_order_request_validation,
    MIN(CASE WHEN h.current_production_stage = 'waiting_fabric_arrival'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_waiting_fabric_arrival,
    MIN(CASE WHEN h.current_production_stage = 'fabric_validation_and_pre_cutting'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_fabric_validation_and_pre_cutting,
    MIN(CASE WHEN h.current_production_stage = 'cut_fabric_and_sewing_process'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_cut_fabric_and_sewing_process,
    MIN(CASE WHEN h.current_production_stage = 'quality_inspection'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_quality_inspection,
    MIN(CASE WHEN h.current_production_stage = 'items_delivery_and_invoicing'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_items_delivery_and_invoicing,
    MIN(CASE WHEN h.current_production_stage = 'finished'
        THEN CAST(h.ingestion_date AS TIMESTAMP) END)
      AS stamp_stage_finished

  FROM `insider-data-lake.sop_silver.supply_chain_efficiency_model_input_history` h
  GROUP BY h.op_code
)
SELECT * EXCEPT(supplier_relationship_status)
FROM CTE_OPS
WHERE dt_planned_entry_warehouse >= '2026-01-01'
  AND (
  supplier_relationship_status IS NULL
  OR supplier_relationship_status NOT IN ('terminated', 'discontinued')
)
"""

print("⏳ Carregando OPs com timestamps de etapas...")
df_ops_raw = client.query(SQL_OPS).to_dataframe()
print(f"✅ df_ops_raw: {len(df_ops_raw):,} linhas | {df_ops_raw.shape[1]} colunas")
df_ops_raw.head(3)

⏳ Carregando OPs com timestamps de etapas...


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


✅ df_ops_raw: 5,284 linhas | 24 colunas


,op_code,current_production_stage,is_finished_product_order,product_names,product_colors,sku_status,supplier_name,cycle_name,production_order_type,planned_quantity_op,...,dt_reviewed_entry_warehouse,dt_largest_entry_warehouse,stamp_created_production_order,stamp_stage_order_request_validation,stamp_stage_waiting_fabric_arrival,stamp_stage_fabric_validation_and_pre_cutting,stamp_stage_cut_fabric_and_sewing_process,stamp_stage_quality_inspection,stamp_stage_items_delivery_and_invoicing,stamp_stage_finished
0,OPF103N4,fabric_delivery_and_validation,True,Camiseta Grafeno Edition Masculino,Preto,ativo_em_lancamento,NATURAL COMPANY CONFECCOES LTDA,GRAFENO,committed,577,...,2026-05-15,2026-04-09,2025-08-13 00:00:00+00:00,NaT,2025-10-29 00:00:00+00:00,2025-11-12 00:00:00+00:00,2025-12-18 00:00:00+00:00,2026-04-01 00:00:00+00:00,2026-04-02 00:00:00+00:00,2026-04-14 00:00:00+00:00
1,OPF87N23,fabric_cutting,True,Calcinha Safe Feminino,Azul Marinho,"desativado,ativo_perene",INTERTEXTIL,D0423C08,committed,1500,...,2026-03-13,2026-03-13,2025-08-13 00:00:00+00:00,2025-11-20 00:00:00+00:00,2025-12-03 00:00:00+00:00,NaT,2026-03-07 00:00:00+00:00,2026-03-04 00:00:00+00:00,2026-03-17 00:00:00+00:00,2026-03-19 00:00:00+00:00
2,OPF103N11,fabric_delivery_and_validation,True,Camiseta Grafeno Edition Masculino,Preto,ativo_em_lancamento,NATURAL COMPANY CONFECCOES LTDA,GRAFENO,committed,577,...,2026-05-15,2026-03-11,2025-08-13 00:00:00+00:00,NaT,2025-10-29 00:00:00+00:00,2025-11-12 00:00:00+00:00,2025-12-18 00:00:00+00:00,2026-02-28 00:00:00+00:00,2026-03-07 00:00:00+00:00,2026-04-03 00:00:00+00:00


In [175]:
# Célula 2.1 — Tempo de produção da matéria-prima principal por produto (Tri)
# Identifica o tecido principal de cada produto e busca o tempo de tingimento
# + produção na malharia. Usado para corrigir o lead time teórico de triangulação.

SQL_FABRIC_TEMPO = """
WITH fabric_costs AS (
    SELECT
        mfs.fabric_id,
        mfs.knitting_factory_id,
        mfs.unit_price,
        mfs.minimum_volume_per_order,
        mf.name AS fabric_name,
        mf.article_id,
        ma.name AS article_name,
        ma.unit AS article_unit,
        mkf.supplier_id,
        ms.alias AS knitting_factory_name
    FROM `insider-data-lake.integrated.muninn_fabric_skus` AS mfs
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mfs.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_knitting_factories` AS mkf ON mkf.id = mfs.knitting_factory_id
    LEFT JOIN `insider-data-lake.integrated.muninn_suppliers` AS ms ON ms.id = mkf.supplier_id
),
fabric_min_max_cost AS (
    SELECT
        fc.fabric_id,
        fc.fabric_name,
        MIN(fc.unit_price) AS min_fabric_cost,
        MAX(fc.unit_price) AS max_fabric_cost,
        MIN(fc.minimum_volume_per_order) AS minimum_volume_per_order,
        COUNT(DISTINCT fc.knitting_factory_id) AS number_knitting_factories,
        ARRAY_AGG(DISTINCT fc.knitting_factory_name) AS knitting_factories_names
    FROM fabric_costs AS fc
    GROUP BY fc.fabric_id, fc.fabric_name
),
skp_with_sales_l8m AS (
    SELECT DISTINCT s.product_name
    FROM `insider-data-lake.fpa.analytical_dre` d
    LEFT JOIN `insider-data-lake.integrated.skus` s USING(sku)
    WHERE DATE(d.order_date) >= DATE_SUB(CURRENT_DATE(), INTERVAL 8 MONTH)
    AND d.order_status != 'Not authorized'
    AND d.quantity > 0
    AND s.product_name IS NOT NULL
),
skp_status AS (
    SELECT
        s.product_name,
        CASE
            WHEN COUNTIF(s.sku_state = 'ativo_perene') > 0         THEN 'ativo_perene'
            WHEN COUNTIF(s.sku_state = 'ativo_em_lancamento') > 0  THEN 'ativo_em_lancamento'
            WHEN COUNTIF(s.sku_state = 'ativo_capsula') > 0        THEN 'ativo_capsula'
            WHEN COUNTIF(s.sku_state = 'personalizacao') > 0       THEN 'personalizacao'
            WHEN COUNTIF(s.sku_state = 'kit') > 0                  THEN 'kit'
            ELSE 'desativado'
        END AS product_status
    FROM `insider-data-lake.integrated.skus` s
    INNER JOIN skp_with_sales_l8m l8m USING(product_name)
    GROUP BY s.product_name
),
sku_fabrics AS (
    SELECT
        s.product_name,
        mpsf.fabric_id,
        mpsf.consumption,
        fc.minimum_volume_per_order,
        ma.unit AS article_unit,
        ma.name AS article_name,
        mf.article_id,
        fc.number_knitting_factories,
        fc.knitting_factories_names
    FROM `insider-data-lake.integrated.muninn_product_skus_fabrics` AS mpsf
    LEFT JOIN `insider-data-lake.integrated.muninn_fabrics` AS mf ON mf.id = mpsf.fabric_id
    LEFT JOIN `insider-data-lake.integrated.muninn_articles` AS ma ON ma.id = mf.article_id
    LEFT JOIN `insider-data-lake.integrated.muninn_product_skus` AS mps ON mps.product_sku_id = mpsf.product_sku_id
    LEFT JOIN `insider-data-lake.integrated.skus` AS s ON mps.sku = s.sku
    LEFT JOIN fabric_min_max_cost AS fc ON fc.fabric_id = mpsf.fabric_id
    INNER JOIN skp_with_sales_l8m l8m ON l8m.product_name = s.product_name
),
product_article_agg AS (
    SELECT
        sf.product_name,
        ss.product_status,
        REGEXP_REPLACE(sf.article_name, r'Modal (\d+)', 'Modal') AS article_name,
        sf.article_unit,
        MIN(sf.minimum_volume_per_order) AS minimum_volume_per_order,
        APPROX_QUANTILES(sf.consumption, 2)[OFFSET(1)] AS median_article_consumption
    FROM sku_fabrics AS sf
    INNER JOIN skp_status AS ss ON ss.product_name = sf.product_name
    WHERE ss.product_status IN ('ativo_perene', 'ativo_em_lancamento', 'desativado')
    AND LOWER(sf.product_name) NOT LIKE '%ziraldo%'
    AND LOWER(sf.product_name) NOT LIKE '% xp%'
    AND LOWER(sf.product_name) NOT LIKE '%maluquinho%'
    AND LOWER(sf.product_name) NOT LIKE '% b2b %'
    GROUP BY sf.product_name, ss.product_status, article_name, sf.article_unit
),
product_main_fabric AS (
    SELECT
        product_name,
        product_status,
        article_name AS tecido_principal,
        article_unit,
        ROUND(CAST(median_article_consumption AS FLOAT64), 4) AS consumo_mediano,
        minimum_volume_per_order
    FROM product_article_agg
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_name
        ORDER BY median_article_consumption DESC
    ) = 1
),
supplier_map AS (
    SELECT
        art.name                        AS artigo,
        akf.coloring_time               AS tempo_tingimento_dias,
        akf.production_time             AS tempo_producao_dias
    FROM `insider-lake-sensitive.prepared_br.prepared_muninn_articles_knitting_factories` akf
    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_knitting_factories`          kf
        ON akf.knitting_factory_id = kf.id
    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_suppliers`                   sup
        ON kf.supplier_id = sup.id
    JOIN `insider-lake-sensitive.prepared_br.prepared_muninn_articles`                    art
        ON akf.article_id = art.id
),
article_final AS (
    SELECT
        REGEXP_REPLACE(artigo, r'Modal (\d+)', 'Modal') AS artigo,
        MAX(tempo_tingimento_dias)                        AS tempo_tingimento,
        MAX(tempo_producao_dias)                          AS tempo_producao_dias,
        MAX(tempo_tingimento_dias) + MAX(tempo_producao_dias) AS tempo_total_dias
    FROM supplier_map
    GROUP BY artigo
)

SELECT
    pmf.product_name,
    pmf.product_status,
    pmf.tecido_principal,
    af.tempo_tingimento,
    af.tempo_producao_dias,
    af.tempo_total_dias
FROM product_main_fabric pmf
LEFT JOIN article_final af
    ON af.artigo = pmf.tecido_principal
ORDER BY pmf.product_name
"""

df_fabric_tempo = client.query(SQL_FABRIC_TEMPO).to_dataframe()
print(f"✅ df_fabric_tempo: {len(df_fabric_tempo)} produtos")
print(f"   Cobertura tempo_total_dias: {df_fabric_tempo['tempo_total_dias'].notna().mean():.1%}")
print(f"   Mediana tempo_total_dias (produtos com cobertura): "
      f"{df_fabric_tempo['tempo_total_dias'].median():.0f}d")


<>:83: SyntaxWarning: invalid escape sequence '\d'
<>:83: SyntaxWarning: invalid escape sequence '\d'
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_6708/3272304854.py:83: SyntaxWarning: invalid escape sequence '\d'
  REGEXP_REPLACE(sf.article_name, r'Modal (\d+)', 'Modal') AS article_name,


✅ df_fabric_tempo: 189 produtos
   Cobertura tempo_total_dias: 83.1%
   Mediana tempo_total_dias (produtos com cobertura): 85d


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [176]:
# Célula 3 — Pré-processamento: filtros, lead time realizado, etapas (6 fases) e mês de fechamento

import numpy as np


# --- 3.1 Filtros conforme metodologia do relatório original ---
df_ops = df_ops_raw.copy()

# Apenas production_order_type = 'committed'
df_ops = df_ops[df_ops["production_order_type"] == "committed"]

# Remover ciclos B2B e EPA
df_ops = df_ops[
    ~df_ops["cycle_name"].str.contains("B2B|EPA", na=False, case=False)
]

# Lead time: metodologia diferenciada por fluxo
# PA: a partir da reserva de MP (waiting_fabric_arrival), excluindo order_request_validation
# TRI: a partir da criação da OP (stamp_created_production_order), incluindo todas as etapas
df_ops["dt_largest_entry_warehouse"] = pd.to_datetime(
    df_ops["dt_largest_entry_warehouse"],
    utc=True,
)

# Cohort de chegada: exclui OPs com data de entrada futura (data planejada, não realizada)
hoje = pd.Timestamp.now(tz="UTC").normalize()
df_ops = df_ops[df_ops["dt_largest_entry_warehouse"] <= hoje]

df_ops["stamp_created_production_order"] = pd.to_datetime(
    df_ops["stamp_created_production_order"],
    utc=True,
)
df_ops["stamp_stage_waiting_fabric_arrival"] = pd.to_datetime(
    df_ops["stamp_stage_waiting_fabric_arrival"],
    utc=True,
)

inicio_lt = df_ops["stamp_stage_waiting_fabric_arrival"].where(
    df_ops["is_finished_product_order"],
    other=df_ops["stamp_created_production_order"],
)

df_ops["lead_time_realizado"] = (
    df_ops["dt_largest_entry_warehouse"] - inicio_lt
).dt.days

# Remover lead times nulos ou negativos
df_ops = df_ops[
    df_ops["lead_time_realizado"].notna()
    & (df_ops["lead_time_realizado"] > 0)
]

print(f"OPs após filtros: {len(df_ops):,}")
print(f"  Triangulação: {(~df_ops['is_finished_product_order']).sum():,}")
print(f"  Produto acabado: {df_ops['is_finished_product_order'].sum():,}")


# --- 3.2 Etapas (6 fases) — mapeadas a partir dos production_stage existentes ---
stamp_cols = [
    "stamp_stage_order_request_validation",
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_quality_inspection",
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_finished",
]

for col in stamp_cols:
    df_ops[col] = pd.to_datetime(df_ops[col], utc=True)


def _diff_days(end_col, start_col):
    """
    Dias entre dois timestamps, com clip(>=0) para não permitir negativos
    por inversão de stamps.
    """
    return (
        (df_ops[end_col] - df_ops[start_col]).dt.total_seconds() / 86400.0
    ).clip(lower=0)


df_ops["etapa_criacao_agd"] = _diff_days(
    "stamp_stage_order_request_validation",
    "stamp_created_production_order",
)
df_ops["etapa_agd_validmp"] = _diff_days(
    "stamp_stage_waiting_fabric_arrival",
    "stamp_stage_order_request_validation",
)
df_ops["etapa_valid_corte"] = _diff_days(
    "stamp_stage_fabric_validation_and_pre_cutting",
    "stamp_stage_waiting_fabric_arrival",
)
df_ops["etapa_valid_corte_exec"] = _diff_days(
    "stamp_stage_cut_fabric_and_sewing_process",
    "stamp_stage_fabric_validation_and_pre_cutting",
)
df_ops["etapa_costura"] = _diff_days(
    "stamp_stage_quality_inspection",
    "stamp_stage_cut_fabric_and_sewing_process",
)
df_ops["etapa_inspecao"] = _diff_days(
    "stamp_stage_items_delivery_and_invoicing",
    "stamp_stage_quality_inspection",
)
df_ops["etapa_fat_estoque"] = _diff_days(
    "dt_largest_entry_warehouse",
    "stamp_stage_items_delivery_and_invoicing",
)

# PA: etapa_criacao_agd não se aplica (LT começa em waiting_fabric_arrival)
df_ops.loc[df_ops["is_finished_product_order"], "etapa_criacao_agd"] = np.nan

ETAPAS_COLS = [
    "etapa_criacao_agd",
    "etapa_agd_validmp",
    "etapa_valid_corte",
    "etapa_valid_corte_exec",
    "etapa_costura",
    "etapa_inspecao",
    "etapa_fat_estoque",
]

ETAPAS_LABELS = [
    "Criação→agd",
    "Agd→valid MP",
    "Valid→corte",
    "Corte exec",
    "Costura",
    "Inspeção",
    "Fat→estoque",
]


# --- 3.3 Mês de fechamento (para séries temporais e janelas móveis) ---
df_ops["mes_fechamento"] = (
    df_ops["dt_largest_entry_warehouse"]
    .dt.tz_convert(None)
    .dt.to_period("M")
    .dt.to_timestamp()
)


# --- 3.4 Enriquecer com lead time teórico da base de capacidade ---
capacity_lt = (
    df_capacity[
        [
            "alias",
            "product_name",
            "is_finished_product",
            "lead_time",
            "tag_abc",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "alias": "supplier_name",
            "lead_time": "lead_time_teorico",
            "is_finished_product": "is_finished_product_order",
        }
    )
)

capacity_lt["is_finished_product_order"] = capacity_lt[
    "is_finished_product_order"
].astype(bool)

df_ops = df_ops.merge(
    capacity_lt,
    how="left",
    left_on=[
        "supplier_name",
        "product_names",
        "is_finished_product_order",
    ],
    right_on=[
        "supplier_name",
        "product_name",
        "is_finished_product_order",
    ],
)


# --- 3.5 Ajuste Tri: somar tempo real de produção da malha ao lead time teórico ---
# Para triangulação, o prazo cadastrado não inclui o tempo de tingimento + produção
# da matéria-prima na malharia. df_fabric_tempo fornece esse valor por produto;
# se não houver mapeamento, aplica fallback de 60d.
FALLBACK_TRI_DIAS = 60

# O primeiro merge (capacity_lt) deixou "product_name" em df_ops; removemos
# antes de fazer o segundo merge para evitar conflito de nome.
if "product_name" in df_ops.columns:
    df_ops = df_ops.drop(columns="product_name")

fabric_tempo_map = (
    df_fabric_tempo[["product_name", "tempo_total_dias"]]
    .drop_duplicates(subset="product_name")
)

df_ops = df_ops.merge(
    fabric_tempo_map,
    left_on="product_names",
    right_on="product_name",
    how="left",
).drop(columns="product_name")

mask_tri = ~df_ops["is_finished_product_order"]

df_ops.loc[mask_tri, "lead_time_teorico"] += df_ops.loc[
    mask_tri,
    "tempo_total_dias",
].fillna(FALLBACK_TRI_DIAS)

cobertura_tri = df_ops.loc[mask_tri, "tempo_total_dias"].notna().mean()
fallback_tri = int((~df_ops.loc[mask_tri, "tempo_total_dias"].notna()).sum())

print(
    f"\n   Ajuste Tri — cobertura tempo_total_dias: {cobertura_tri:.1%} "
    f"| fallback {FALLBACK_TRI_DIAS}d aplicado em {fallback_tri} OPs"
)

df_ops["desvio_lt"] = (
    df_ops["lead_time_realizado"] - df_ops["lead_time_teorico"]
)
df_ops["dentro_do_prazo"] = (
    df_ops["lead_time_realizado"] <= TARGET_LEAD_TIME
)


# --- 3.6 Buckets de volume (consome CONFIG quando definido; fallback para defaults) ---
_bins = (
    CONFIG["buckets_volume"]
    if "CONFIG" in globals()
    else [0, 200, 500, 1000, 2000, 5000, float("inf")]
)
_labels = (
    CONFIG["labels_volume"]
    if "CONFIG" in globals()
    else [
        "<200",
        "200-499",
        "500-999",
        "1000-1999",
        "2000-4999",
        "5000+",
    ]
)

df_ops["volume_bucket"] = pd.cut(
    df_ops["planned_quantity_op"],
    bins=_bins,
    labels=_labels,
    right=False,
)


# --- 3.7 Separar fluxos (re-derivado APÓS todas as colunas calculadas) ---
df_tri = df_ops[~df_ops["is_finished_product_order"]].copy()
df_pa = df_ops[df_ops["is_finished_product_order"]].copy()

print("\n✅ Pré-processamento completo.")
print(f"   df_tri (triangulação): {len(df_tri):,} OPs")
print(f"   df_pa  (produto acabado): {len(df_pa):,} OPs")
print(
    "   Cobertura lead time teórico: "
    f"{df_ops['lead_time_teorico'].notna().mean():.1%}"
)
print(
    "   Cobertura das 6 etapas (todas preenchidas): "
    f"{df_ops[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)
print(
    f"     ↳ PA:  {df_pa[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)
print(
    f"     ↳ Tri: {df_tri[ETAPAS_COLS].notna().all(axis=1).mean():.1%}"
)

OPs após filtros: 857
  Triangulação: 162
  Produto acabado: 695

   Ajuste Tri — cobertura tempo_total_dias: 89.5% | fallback 60d aplicado em 17 OPs

✅ Pré-processamento completo.
   df_tri (triangulação): 162 OPs
   df_pa  (produto acabado): 695 OPs
   Cobertura lead time teórico: 75.1%
   Cobertura das 6 etapas (todas preenchidas): 8.1%
     ↳ PA:  0.0%
     ↳ Tri: 42.6%


In [177]:
# Célula 4 — Nível 1: KPI Cards (com variação MoM)
from IPython.display import display, HTML

# Valor atual (último mês fechado completo)
mes_atual = df_ops["mes_fechamento"].max()
mes_anterior = (mes_atual - pd.DateOffset(months=1))


def mediana_mes(df, mes):
    sub = df[df["mes_fechamento"] == mes]
    return sub["lead_time_realizado"].median() if len(sub) >= 10 else None


lt_geral_atual = df_ops["lead_time_realizado"].median()
lt_tri_atual   = df_tri["lead_time_realizado"].median()
lt_pa_atual    = df_pa["lead_time_realizado"].median()
pct_dentro     = df_ops["dentro_do_prazo"].mean() * 100

# Variação MoM
lt_geral_ant = mediana_mes(df_ops, mes_anterior)
lt_tri_ant   = mediana_mes(df_tri, mes_anterior)
lt_pa_ant    = mediana_mes(df_pa,  mes_anterior)


def delta_mom(atual, anterior):
    if anterior is None or pd.isna(anterior) or pd.isna(atual):
        return ""
    delta = atual - anterior
    arrow = "▼" if delta < 0 else ("▲" if delta > 0 else "→")
    color = COLORS["status_ok"] if delta < 0 else (COLORS["status_critico"] if delta > 0 else COLORS["neutro_escuro"])
    return f"<span style='color:{color}'>{arrow} {abs(delta):.0f}d MoM</span>"


def kpi_card(label, value, unit="", color="#374151", sub=None):
    sub_html = f"<div style='font-size:12px;color:#666;margin-top:6px'>{sub}</div>" if sub else ""
    return f"""
    <div style='display:inline-block;background:#fafafa;border:1px solid #e5e7eb;
                border-radius:10px;padding:18px 26px;margin:8px;min-width:170px;text-align:center'>
        <div style='font-size:12px;color:#6b7280;font-weight:600;text-transform:uppercase;letter-spacing:0.5px'>{label}</div>
        <div style='font-size:34px;font-weight:700;color:{color};margin-top:4px'>{value}<span style='font-size:14px;font-weight:500'>{unit}</span></div>
        {sub_html}
    </div>"""


# Cor de status para %120d
if pct_dentro >= 65:
    cor_pct = COLORS["status_ok"]
elif pct_dentro >= 50:
    cor_pct = COLORS["status_atencao"]
else:
    cor_pct = COLORS["status_critico"]

cards_html = "".join([
    kpi_card("Mediana Geral",        f"{lt_geral_atual:.0f}", "d", COLORS["neutro_escuro"],
             f"Target: {TARGET_LEAD_TIME}d &nbsp;|&nbsp; {delta_mom(lt_geral_atual, lt_geral_ant)}"),
    kpi_card("Mediana Triangulação", f"{lt_tri_atual:.0f}",   "d", COLORS["fluxo_tri"],
             delta_mom(lt_tri_atual, lt_tri_ant)),
    kpi_card("Mediana Produto Acabado", f"{lt_pa_atual:.0f}", "d", COLORS["fluxo_pa"],
             delta_mom(lt_pa_atual, lt_pa_ant)),
    kpi_card("Dentro do Prazo (120d)", f"{pct_dentro:.1f}", "%", cor_pct,
             f"Janela: últimos {CONFIG['janela_meses']}m"),
])
display(HTML(f"""
<div style='font-family:-apple-system,sans-serif'>
    <h3 style='color:#374151;margin-bottom:8px'>📊 Nível 1 — Visão Executiva</h3>
    {cards_html}
</div>
"""))


In [178]:
# Célula 5 — Nível 1: Série Temporal Mensal por Fluxo
# Substitui o snapshot Mediana×P75 estático: o último ponto da série já entrega o snapshot
# e ainda dá tendência. Faixa P25–P75 sombreada + barras de %120d no eixo secundário.

# Cortar pela janela configurada
data_corte = pd.Timestamp.now(tz="UTC").normalize() - pd.DateOffset(months=CONFIG["janela_meses"])
df_ts = df_ops[df_ops["mes_fechamento"] >= data_corte.tz_localize(None)].copy()


def serie_por_fluxo(df_subset):
    agg = (
        df_subset.groupby("mes_fechamento")
        .agg(
            mediana=("lead_time_realizado", "median"),
            p75=("lead_time_realizado", lambda x: x.quantile(0.75)),
            p90=("lead_time_realizado", lambda x: x.quantile(0.90)),
            pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            n_ops=("op_code", "count"),
        )
        .reset_index()
    )
    # Filtro de robustez: meses com < 20 OPs viram NaN
    agg.loc[agg["n_ops"] < 20, ["mediana", "p75", "p90", "pct_no_prazo"]] = None
    return agg


ts_tri = serie_por_fluxo(df_ts[~df_ts["is_finished_product_order"]])
ts_pa  = serie_por_fluxo(df_ts[df_ts["is_finished_product_order"]])

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Triangulação", "Produto Acabado"),
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    horizontal_spacing=0.12,
)


def add_fluxo(fig, ts, col, cor_fluxo, nome):
    # Faixa P25–P75 (sombreada)
    fig.add_trace(go.Scatter(
        x=list(ts["mes_fechamento"]) + list(ts["mes_fechamento"])[::-1],
        y=list(ts["p90"]) + list(ts["p75"])[::-1],
        fill="toself", fillcolor=cor_fluxo, opacity=0.15,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
        name=f"P75–P90 {nome}",
    ), row=1, col=col, secondary_y=False)

    # Linha mediana (forte)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["mediana"], mode="lines+markers",
        line=dict(color=cor_fluxo, width=3), marker=dict(size=8),
        name=f"Mediana {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Linha P75 (pontilhada)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["p75"], mode="lines",
        line=dict(color=cor_fluxo, width=1.5, dash="dot"),
        name=f"P75 {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=False)

    # Linha P90 (tracejada)
    fig.add_trace(go.Scatter(
        x=ts["mes_fechamento"], y=ts["p90"], mode="lines+markers",
        line=dict(color=cor_fluxo, width=1.5, dash="dash"),
        marker=dict(size=6, symbol="diamond"),
        name=f"P90 {nome}", legendgroup=nome,
        hovertemplate="%{x|%b %Y}<br>P90: <b>%{y:.0f}d</b><extra></extra>",
    ), row=1, col=col, secondary_y=False)

    # Barras %dentro120d no eixo secundário
    fig.add_trace(go.Bar(
        x=ts["mes_fechamento"], y=ts["pct_no_prazo"],
        marker_color=cor_fluxo, opacity=0.25,
        name=f"% no prazo {nome}", legendgroup=nome,
    ), row=1, col=col, secondary_y=True)

    # Linha target 120d
    fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash",
                  line_color=COLORS["target_line"], row=1, col=col, secondary_y=False)

    # Anotação da variação MoM no último ponto válido
    validos = ts.dropna(subset=["mediana"])
    if len(validos) >= 2:
        ultimo = validos.iloc[-1]
        penultimo = validos.iloc[-2]
        delta = ultimo["mediana"] - penultimo["mediana"]
        arrow = "▼" if delta < 0 else "▲"
        cor_anot = COLORS["status_ok"] if delta < 0 else COLORS["status_critico"]
        fig.add_annotation(
            x=ultimo["mes_fechamento"], y=ultimo["mediana"],
            text=f"{arrow} {abs(delta):.0f}d MoM",
            showarrow=True, arrowhead=2, ax=30, ay=-30,
            font=dict(color=cor_anot, size=11, family="sans-serif"),
            row=1, col=col,
        )


add_fluxo(fig, ts_tri, col=1, cor_fluxo=COLORS["fluxo_tri"], nome="Tri")
add_fluxo(fig, ts_pa,  col=2, cor_fluxo=COLORS["fluxo_pa"],  nome="PA")

fig.update_xaxes(title_text="Mês", row=1, col=1)
fig.update_xaxes(title_text="Mês", row=1, col=2)
fig.update_yaxes(title_text="Dias", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Dias", row=1, col=2, secondary_y=False)
fig.update_yaxes(title_text="% no prazo", row=1, col=1, secondary_y=True, range=[0, 100])
fig.update_yaxes(title_text="% no prazo", row=1, col=2, secondary_y=True, range=[0, 100])

fig.update_layout(
    title=f"Evolução do Lead Time — últimos {CONFIG['janela_meses']} meses",
    template=TEMPLATE, height=480,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
)
fig.show()


In [179]:
# Célula 6 — Nível 2: Decomposição de Lead Time por Etapa (6 etapas, PA e Tri)
# Parâmetro: trocar para "PA", "Tri" ou "Ambos"
FLUXO_FILTRO = "Ambos"

# Escala sequencial para as 6 etapas (gradient azul claro → escuro)
ETAPAS_CORES = ["#DBEAFE", "#93C5FD", "#60A5FA", "#2563EB", "#3B82F6", "#1D4ED8", "#1E3A8A"]


def montar_etapas(df_subset, label_fluxo):
    if len(df_subset) == 0:
        return None
    agg = (
        df_subset.groupby("supplier_name")
        .agg(
            **{col: (col, "median") for col in ETAPAS_COLS},
            n_ops=("op_code", "count"),
            lt_total=("lead_time_realizado", "median"),
        )
        .reset_index()
    )
    agg = agg[agg["n_ops"] >= CONFIG["min_ops_grafico"]]
    agg = agg.nlargest(CONFIG["top_n_fornecedores_grafico"], "n_ops")
    agg["fluxo"] = label_fluxo
    # Ordenar por lt_total ascendente → maior LDT no topo do gráfico (barh)
    agg = agg.sort_values("lt_total", ascending=True)
    return agg


dfs_para_plotar = []
if FLUXO_FILTRO in ("PA", "Ambos"):
    pa_etapas = montar_etapas(df_pa, "PA")
    if pa_etapas is not None:
        dfs_para_plotar.append(pa_etapas)
if FLUXO_FILTRO in ("Tri", "Ambos"):
    tri_etapas = montar_etapas(df_tri, "Tri")
    if tri_etapas is not None:
        dfs_para_plotar.append(tri_etapas)

if not dfs_para_plotar:
    print(f"⚠ Sem dados suficientes para o filtro FLUXO_FILTRO={FLUXO_FILTRO!r}")
else:
    n_subplots = len(dfs_para_plotar)
    fig = make_subplots(
        rows=1, cols=n_subplots,
        subplot_titles=[f"{d['fluxo'].iloc[0]} (top {len(d)} por volume)" for d in dfs_para_plotar],
        shared_yaxes=False,
        horizontal_spacing=0.18,
    )

    for idx, agg in enumerate(dfs_para_plotar, start=1):
        for i, (col, label) in enumerate(zip(ETAPAS_COLS, ETAPAS_LABELS)):
            if agg[col].isna().all():
                continue
            fig.add_trace(go.Bar(
                y=agg["supplier_name"],
                x=agg[col],
                name=label,
                orientation="h",
                marker_color=ETAPAS_CORES[i],
                text=[f"{v:.0f}d" if v >= 10 else "" for v in agg[col]],
                textposition="inside",
                insidetextanchor="middle",
                textfont=dict(size=10, color="white" if i >= 3 else "#374151"),
                hovertemplate=f"<b>%{{y}}</b><br>{label}: %{{x:.0f}}d<extra></extra>",
                showlegend=(idx == 1),
                legendgroup=label,
            ), row=1, col=idx)

        fig.add_vline(x=TARGET_LEAD_TIME, line_dash="dash",
                      line_color=COLORS["target_line"], row=1, col=idx)
        fig.update_xaxes(title_text="Dias (mediana)", row=1, col=idx)

    fig.update_layout(
        title=f"Decomposição de Lead Time por Etapa — {FLUXO_FILTRO}",
        barmode="stack", template=TEMPLATE,
        height=max(450, 30 * max(len(d) for d in dfs_para_plotar) + 100),
        legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
    )
    fig.show()


In [180]:
# Célula 6.1 — Tabela detalhada de etapas: Fornecedor × Produto × Fluxo
# Reproduz o formato da tabela "Gargalo por etapa" da guilda de LDT.

PRODUTO_FILTRO = None         # None = todos | ou string parcial (ex: "Tech T-shirt")
FLUXO_FILTRO_TABELA = "Ambos" # "PA", "Tri" ou "Ambos"
MIN_OPS_TABELA = 3

_agg_spec = {col: (col, "median") for col in ETAPAS_COLS}
_agg_spec["n_ops"] = ("op_code", "count")
_agg_spec["lt_total"] = ("lead_time_realizado", "median")

base = df_ops.copy()
if FLUXO_FILTRO_TABELA == "PA":
    base = base[base["is_finished_product_order"] == True]
elif FLUXO_FILTRO_TABELA == "Tri":
    base = base[base["is_finished_product_order"] == False]
if PRODUTO_FILTRO:
    base = base[base["product_names"].str.contains(PRODUTO_FILTRO, case=False, na=False)]

tabela = (
    base.groupby(["supplier_name", "product_names", "is_finished_product_order"], dropna=False)
    .agg(**_agg_spec)
    .reset_index()
)
tabela = tabela[tabela["n_ops"] >= MIN_OPS_TABELA].copy()
tabela["fluxo"] = tabela["is_finished_product_order"].map({True: "PA", False: "Tri"})

# Identificar gargalo principal por linha
tabela["gargalo_etapa"] = tabela[ETAPAS_COLS].idxmax(axis=1).map(dict(zip(ETAPAS_COLS, ETAPAS_LABELS)))
tabela["gargalo_dias"] = tabela[ETAPAS_COLS].max(axis=1)
tabela["gargalo_principal"] = (
    tabela["gargalo_etapa"] + " (" + tabela["gargalo_dias"].round(0).astype("Int64").astype(str) + "d)"
)

# Ordenar por lt_total decrescente
tabela = tabela.sort_values("lt_total", ascending=False)

# Truncar nome de produto
tabela["produto"] = tabela["product_names"].astype(str).str[:35]

# Renomear colunas de etapas para os labels finais
rename_map = dict(zip(ETAPAS_COLS, ETAPAS_LABELS))
tabela = tabela.rename(columns=rename_map)

display_cols = ["supplier_name", "produto", "fluxo", "n_ops", "lt_total",
                *ETAPAS_LABELS, "gargalo_principal"]


def color_fluxo(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""


print(f"Tabela: {len(tabela)} pares (fornecedor × produto × fluxo) — filtro: {FLUXO_FILTRO_TABELA}, mín {MIN_OPS_TABELA} OPs")
(tabela[display_cols]
    .style
    .background_gradient(subset=ETAPAS_LABELS, cmap="Blues", axis=None)
    .map(color_fluxo, subset=["fluxo"])
    .format({
        "lt_total": "{:.0f}d",
        **{l: "{:.0f}d" for l in ETAPAS_LABELS[1:]},
        ETAPAS_LABELS[0]: lambda x: "N/A" if pd.isna(x) else f"{x:.0f}d",
    })
    .hide(axis="index")
)


Tabela: 87 pares (fornecedor × produto × fluxo) — filtro: Ambos, mín 3 OPs


supplier_name,produto,fluxo,n_ops,lt_total,Criação→agd,Agd→valid MP,Valid→corte,Corte exec,Costura,Inspeção,Fat→estoque,gargalo_principal
NATURAL COMPANY CONFECCOES LTDA,Camiseta Grafeno Edition Feminino,PA,4,201d,N/A,nand,14d,36d,132d,14d,4d,Costura (132d)
WARUSKY,"Calça Wide Leg InLounge Feminino,Ca",PA,3,191d,N/A,nand,75d,101d,6d,9d,0d,Corte exec (101d)
LUTESTIL,Calcinha Minimal Corte a Laser Femi,PA,10,187d,N/A,0d,24d,104d,56d,3d,1d,Corte exec (104d)
NOVA FORMULA,Camiseta Henley Core Masculino,Tri,4,170d,0d,32d,40d,47d,34d,4d,3d,Corte exec (47d)
NATURAL COMPANY CONFECCOES LTDA,Camiseta Polo Core Masculino,Tri,5,165d,0d,33d,38d,16d,67d,1d,1d,Costura (67d)
ART LIVRE,Tech T-shirt Heavy Slim Masculino,PA,4,164d,N/A,14d,102d,39d,9d,9d,4d,Valid→corte (102d)
LUTESTIL,Undershirt Anti Suor Gola V Masculi,PA,3,159d,N/A,nand,43d,74d,41d,6d,1d,Corte exec (74d)
RIZLLEP,Saia Envelope Breeze Feminino,PA,7,154d,N/A,33d,30d,39d,55d,25d,6d,Costura (55d)
ART LIVRE,Tech T-shirt Gola U Masculino,Tri,3,153d,0d,20d,78d,14d,26d,14d,3d,Valid→corte (78d)
MC & MC,Camisa FutureForm Masculino,PA,3,150d,N/A,66d,104d,20d,24d,5d,3d,Valid→corte (104d)


In [181]:
# Célula 8.1 — Nível 1: LDT mediano por bucket de volume × fluxo (agregado executivo)
matrix = (
    df_ops.groupby(["volume_bucket", "is_finished_product_order"], observed=True)["lead_time_realizado"]
    .agg(["median", "count"])
    .reset_index()
)
matrix.columns = ["volume_bucket", "is_finished_product_order", "mediana", "n_ops"]
matrix["fluxo"] = matrix["is_finished_product_order"].map({True: "Produto Acabado", False: "Triangulação"})
matrix = matrix[matrix["n_ops"] >= CONFIG["min_ops_grafico"]]

fig = px.bar(
    matrix, x="volume_bucket", y="mediana", color="fluxo", barmode="group",
    text=matrix.apply(lambda r: f"{r['mediana']:.0f}d (n={r['n_ops']})" if r['n_ops'] >= 10 else f"{r['mediana']:.0f}d", axis=1),
    color_discrete_map={
        "Triangulação": COLORS["fluxo_tri"],
        "Produto Acabado": COLORS["fluxo_pa"],
    },
    labels={"mediana": "Lead Time Mediano (dias)", "volume_bucket": "Faixa de Volume"},
    title="LDT Mediano por Faixa de Volume × Fluxo",
    template=TEMPLATE,
    category_orders={"volume_bucket": CONFIG["labels_volume"]},
)
fig.add_hline(y=TARGET_LEAD_TIME, line_dash="dash", line_color=COLORS["target_line"],
              annotation_text=f"Target {TARGET_LEAD_TIME}d")
fig.update_traces(textposition="outside")
fig.update_layout(height=420)
fig.show()


In [182]:
# Célula 8.2 — Nível 2: Heatmap Fornecedor × Faixa de Volume (PA e Tri lado a lado)


def montar_heatmap(df_subset, label):
    top_forn = (
        df_subset.groupby("supplier_name")["op_code"].count()
        .nlargest(CONFIG["top_n_fornecedores_heatmap"]).index.tolist()
    )
    pivot_mediana = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["lead_time_realizado"]
        .median().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        df_subset[df_subset["supplier_name"].isin(top_forn)]
        .groupby(["supplier_name", "volume_bucket"], observed=True)["op_code"]
        .count().unstack("volume_bucket")
        .reindex(columns=CONFIG["labels_volume"])
    )
    # Mascarar células com n_ops < min_ops_grafico
    pivot_mediana = pivot_mediana.where(pivot_n >= CONFIG["min_ops_grafico"])
    # Ordenar por mediana geral do fornecedor (asc = melhor no topo)
    ordem = pivot_mediana.median(axis=1).sort_values(ascending=True).index
    return pivot_mediana.loc[ordem], pivot_n.loc[ordem], label


hm_tri, n_tri, _ = montar_heatmap(df_tri, "Triangulação")
hm_pa,  n_pa,  _ = montar_heatmap(df_pa,  "Produto Acabado")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Triangulação", "Produto Acabado"),
                    horizontal_spacing=0.18)

for idx, (hm, n_df, _) in enumerate([(hm_tri, n_tri, "Tri"), (hm_pa, n_pa, "PA")], start=1):
    texto = hm.copy().astype(object)
    for i in hm.index:
        for j in hm.columns:
            v = hm.loc[i, j]
            n_val = n_df.loc[i, j] if not pd.isna(n_df.loc[i, j]) else 0
            texto.loc[i, j] = f"{v:.0f}d<br>n={int(n_val)}" if pd.notna(v) else "—"

    fig.add_trace(go.Heatmap(
        z=hm.values, x=list(hm.columns), y=list(hm.index),
        text=texto.values, texttemplate="%{text}",
        textfont=dict(size=10),
        colorscale=[
            [0.0,  COLORS["status_ok"]],
            [0.4,  "#FEF3C7"],
            [0.5,  COLORS["target_line"]],
            [0.6,  COLORS["status_atencao"]],
            [1.0,  COLORS["status_critico"]],
        ],
        zmid=TARGET_LEAD_TIME,
        zmin=30, zmax=210,
        showscale=(idx == 2),
        colorbar=dict(title="LDT (d)", x=1.02) if idx == 2 else None,
        hovertemplate="<b>%{y}</b><br>Volume: %{x}<br>LDT: %{z:.0f}d<extra></extra>",
    ), row=1, col=idx)

fig.update_layout(
    title="Heatmap: Lead Time Mediano por Fornecedor × Faixa de Volume",
    template=TEMPLATE,
    height=max(500, 25 * max(len(hm_tri), len(hm_pa)) + 100),
)
fig.update_xaxes(title_text="Faixa de Volume")
fig.show()


In [183]:
# Célula 8.5 — Nível 3: Tabela de Recomendação de Prazo por Fornecedor × Produto × Fluxo
# Output direto da ação P1 da guilda: substituir o prazo padrão de 120d pelo prazo real recomendado.
# Granularidade: fornecedor × product_names × fluxo (PA / Tri)

PERCENTIL = CONFIG["percentil_recomendacao"]
JANELA_LONGA = CONFIG["janela_meses"]
JANELA_CURTA = CONFIG["janela_meses_curta"]

MIN_OPS_REC = 3  # mínimo local — mais permissivo que o global de 5,
                 # pois granularidade por produto reduz n por célula


def recomendacao_por_produto(df_subset, janela_meses, min_ops=MIN_OPS_REC):
    data_corte = (
        pd.Timestamp.now(tz="UTC").normalize()
        - pd.DateOffset(months=janela_meses)
    ).tz_localize(None)

    sub = df_subset[df_subset["mes_fechamento"] >= data_corte].copy()

    agg = (
        sub.groupby(
            [
                "supplier_name",
                "product_names",
                "is_finished_product_order",
            ]
        )
        .agg(
            n_ops=("op_code", "count"),
            lt_realizado_p50=("lead_time_realizado", "median"),
            lt_recomendado_raw=(
                "lead_time_realizado",
                lambda x: x.quantile(PERCENTIL),
            ),
            pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
            lt_teorico_cadastrado=("lead_time_teorico", "median"),
        )
        .reset_index()
    )

    agg = agg[agg["n_ops"] >= min_ops].copy()
    agg["lt_recomendado"] = (agg["lt_recomendado_raw"] / 5).round() * 5

    return agg.drop(columns="lt_recomendado_raw")


rec_longa = recomendacao_por_produto(df_ops, JANELA_LONGA)

rec_curta = (
    recomendacao_por_produto(df_ops, JANELA_CURTA)[
        [
            "supplier_name",
            "product_names",
            "is_finished_product_order",
            "lt_recomendado",
            "n_ops",
        ]
    ]
    .rename(
        columns={
            "lt_recomendado": "lt_recomendado_curta",
            "n_ops": "n_ops_curta",
        }
    )
)

rec = rec_longa.merge(
    rec_curta,
    on=[
        "supplier_name",
        "product_names",
        "is_finished_product_order",
    ],
    how="left",
)

rec["fluxo"] = rec["is_finished_product_order"].map(
    {
        True: "PA",
        False: "Tri",
    }
)

# Ajuste Tri já incorporado em lead_time_teorico (Cell 3 / Célula 2.1)
rec["delta_vs_cadastrado"] = (
    rec["lt_recomendado"] - rec["lt_teorico_cadastrado"]
).round(0)


def direcao(d):
    if pd.isna(d):
        return "—"

    if d > 10:
        return "▲ Aumentar"

    if d < -10:
        return "▼ Reduzir"

    return "→ Manter"


def confianca(n):
    if n >= 30:
        return "🟢 Alta"

    if n >= 10:
        return "🟡 Média"

    return "🔴 Baixa"


def tendencia(row):
    if (
        pd.isna(row.get("lt_recomendado_curta"))
        or row.get("n_ops_curta", 0) < 3
    ):
        return "—"

    diff = row["lt_recomendado_curta"] - row["lt_recomendado"]

    if diff < -5:
        return "📉 Melhorando"

    if diff > 5:
        return "📈 Piorando"

    return "≡ Estável"


rec["direcao"] = rec["delta_vs_cadastrado"].apply(direcao)
rec["confianca"] = rec["n_ops"].apply(confianca)
rec["tendencia_3m"] = rec.apply(tendencia, axis=1)

rec["produto"] = rec["product_names"].str[:40]

rec = rec.sort_values(
    "delta_vs_cadastrado",
    key=lambda x: x.abs(),
    ascending=False,
)

display_cols = [
    "supplier_name",
    "produto",
    "fluxo",
    "n_ops",
    "lt_teorico_cadastrado",
    "lt_recomendado",
    "lt_recomendado_curta",
    "delta_vs_cadastrado",
    "direcao",
    "pct_no_prazo",
    "tendencia_3m",
    "confianca",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (
            f"background-color:{COLORS['fluxo_pa']};"
            "color:white; font-weight:600; text-align:center"
        )

    if v == "Tri":
        return (
            f"background-color:{COLORS['fluxo_tri']};"
            "color:#374151; font-weight:600; text-align:center"
        )

    return ""


def color_direcao(v):
    if "Aumentar" in str(v):
        return (
            f"background-color:{COLORS['status_critico']}; "
            "color:white; font-weight:600"
        )

    if "Reduzir" in str(v):
        return (
            f"background-color:{COLORS['status_ok']}; "
            "color:white; font-weight:600"
        )

    return ""


print(
    f"📋 Recomendação de Prazo — P{int(PERCENTIL * 100)} | "
    f"janela {JANELA_LONGA}m "
    f"(curta: {JANELA_CURTA}m) | "
    f"mín {MIN_OPS_REC} OPs por par"
)
print("   Granularidade: fornecedor × produto × fluxo")
print(f"   Pares com amostra suficiente: {len(rec)}")
print(
    f"   ▲ Aumentar: {(rec['direcao'] == '▲ Aumentar').sum()} | "
    f"▼ Reduzir: {(rec['direcao'] == '▼ Reduzir').sum()} | "
    f"→ Manter: {(rec['direcao'] == '→ Manter').sum()}"
)

(
    rec[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .map(color_direcao, subset=["direcao"])
    .background_gradient(
        subset=["delta_vs_cadastrado"],
        cmap="RdYlGn_r",
        vmin=-40,
        vmax=90,
    )
    .format(
        {
            "lt_teorico_cadastrado": "{:.0f}d",
            "lt_recomendado": "{:.0f}d",
            "lt_recomendado_curta": "{:.0f}d",
            "delta_vs_cadastrado": "{:+.0f}d",
            "pct_no_prazo": "{:.1f}%",
        },
        na_rep="—",
    )
    .hide(axis="index")
)

📋 Recomendação de Prazo — P75 | janela 12m (curta: 3m) | mín 3 OPs por par
   Granularidade: fornecedor × produto × fluxo
   Pares com amostra suficiente: 87
   ▲ Aumentar: 45 | ▼ Reduzir: 2 | → Manter: 10


supplier_name,produto,fluxo,n_ops,lt_teorico_cadastrado,lt_recomendado,lt_recomendado_curta,delta_vs_cadastrado,direcao,pct_no_prazo,tendencia_3m,confianca
NATURAL COMPANY CONFECCOES LTDA,Camiseta Grafeno Edition Feminino,PA,4,90d,200d,200d,+110d,▲ Aumentar,0.0%,≡ Estável,🔴 Baixa
BAE BRASIL,NoHo Socks,PA,9,45d,130d,135d,+85d,▲ Aumentar,66.7%,≡ Estável,🔴 Baixa
LUTESTIL,Undershirt Anti Suor Gola V Masculino,PA,3,90d,170d,170d,+80d,▲ Aumentar,0.0%,≡ Estável,🔴 Baixa
MALHAS D'STEFANO,Spectrum Socks Mid 2.0,PA,21,60d,135d,175d,+75d,▲ Aumentar,66.7%,📈 Piorando,🟡 Média
MALHAS D'STEFANO,Spectrum Socks Low 2.0,PA,15,60d,135d,160d,+75d,▲ Aumentar,66.7%,📈 Piorando,🟡 Média
ART LIVRE,Tech T-shirt Heavy Slim Masculino,PA,4,100d,175d,175d,+75d,▲ Aumentar,0.0%,≡ Estável,🔴 Baixa
BAE BRASIL,Cueca Boxer Performance Simples Masculin,PA,5,75d,150d,155d,+75d,▲ Aumentar,40.0%,≡ Estável,🔴 Baixa
BAE BRASIL,The Perfect Top Feminino,PA,66,75d,145d,155d,+70d,▲ Aumentar,62.1%,📈 Piorando,🟢 Alta
MALHAS D'STEFANO,Spectrum Socks High 2.0,PA,23,60d,130d,160d,+70d,▲ Aumentar,60.9%,📈 Piorando,🟡 Média
MC & MC,Camisa FutureForm Masculino,PA,3,90d,150d,150d,+60d,▲ Aumentar,33.3%,≡ Estável,🔴 Baixa


In [184]:
# Célula 9 — Nível 3: Risco Single-Source (matriz + tabela fornecedor × produto)

# Identificar produtos single-source
single_source_produtos = df_capacity[df_capacity["num_suppliers_per_product"] == 1][
    ["product_name", "alias", "is_finished_product", "lead_time", "tag_abc"]
].drop_duplicates().rename(columns={"alias": "fornecedor", "lead_time": "lt_teorico"})

# Enriquecer com dados de OPs (lead time realizado, desvio, % no prazo)
ops_por_par = (
    df_ops.groupby(["product_names", "supplier_name", "is_finished_product_order"])
    .agg(
        lt_realizado=("lead_time_realizado", "median"),
        desvio_mediano=("desvio_lt", "median"),
        pct_no_prazo=("dentro_do_prazo", lambda x: x.mean() * 100),
        n_ops=("op_code", "count"),
    )
    .reset_index()
    .rename(columns={
        "product_names": "product_name",
        "supplier_name": "fornecedor",
        "is_finished_product_order": "is_finished_product",
    })
)

single_source = single_source_produtos.merge(
    ops_por_par, how="left",
    on=["product_name", "fornecedor", "is_finished_product"],
)
single_source["fluxo"] = single_source["is_finished_product"].map({True: "PA", False: "Tri"})


def nivel(row):
    if row.get("tag_abc") in ["A", "B"]:
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_critico"]:
            return "🔴 Alto"
        if pd.notna(row.get("desvio_mediano")) and row["desvio_mediano"] > CONFIG["threshold_desvio_atencao"]:
            return "🟠 Moderado"
        return "🟡 Acompanhar"
    return "🟢 Baixo"


single_source["nivel_atencao"] = single_source.apply(nivel, axis=1)

# ====== MATRIZ DE RISCO (scatter) ======
matriz = single_source.dropna(subset=["desvio_mediano", "n_ops"]).copy()

simbolo_abc = {"A": "circle", "B": "square", "C": "diamond"}
matriz["simbolo"] = matriz["tag_abc"].map(simbolo_abc).fillna("x")

fig = go.Figure()

for abc in ["A", "B", "C"]:
    sub = matriz[matriz["tag_abc"] == abc]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=sub["n_ops"], y=sub["desvio_mediano"],
        mode="markers",
        marker=dict(
            symbol=simbolo_abc[abc],
            size=sub["n_ops"].clip(5, 50),
            color=sub["pct_no_prazo"],
            colorscale=[[0.0, COLORS["status_critico"]],
                        [0.5, COLORS["status_atencao"]],
                        [1.0, COLORS["status_ok"]]],
            cmin=0, cmax=100,
            colorbar=dict(title="% dentro 120d", x=1.02) if abc == "A" else None,
            showscale=(abc == "A"),
            line=dict(width=1, color="#374151"),
        ),
        name=f"Tag {abc}",
        text=sub["fornecedor"].astype(str) + " — " + sub["product_name"].astype(str).str[:30] + " (" + sub["fluxo"].astype(str) + ")",
        hovertemplate=(
            "<b>%{text}</b><br>"
            "n_OPs: %{x}<br>"
            "Desvio: %{y:.0f}d<br>"
            f"Tag ABC: {abc}<extra></extra>"
        ),
    ))

fig.add_hline(y=CONFIG["threshold_desvio_critico"], line_dash="dash",
              line_color=COLORS["status_critico"], annotation_text="Crítico >30d")
fig.add_hline(y=CONFIG["threshold_desvio_atencao"], line_dash="dot",
              line_color=COLORS["status_atencao"], annotation_text="Atenção >10d")
fig.add_hline(y=0, line_color="#374151", line_width=0.5)

fig.update_layout(
    title="Matriz de Risco Single-Source — Par Fornecedor × Produto",
    template=TEMPLATE,
    xaxis=dict(title="Volume (n_OPs últimos 12m) — log", type="log"),
    yaxis=dict(title="Desvio mediano vs teórico (dias)"),
    height=520,
    legend=dict(title="Criticidade ABC"),
)
fig.show()

# ====== TABELA fornecedor × produto ======
display_cols = ["product_name", "fornecedor", "fluxo", "tag_abc",
                "lt_teorico", "lt_realizado", "desvio_mediano", "pct_no_prazo",
                "n_ops", "nivel_atencao"]


def color_fluxo_cell(v):
    if v == "PA":
        return f"background-color:{COLORS['fluxo_pa']}; color:white; font-weight:600; text-align:center"
    if v == "Tri":
        return f"background-color:{COLORS['fluxo_tri']}; color:#374151; font-weight:600; text-align:center"
    return ""


def color_atencao(v):
    if "Alto" in str(v):
        return "font-weight:600"
    return ""


ordem_atencao = {"🔴 Alto": 0, "🟠 Moderado": 1, "🟡 Acompanhar": 2, "🟢 Baixo": 3}
single_source["_ord"] = single_source["nivel_atencao"].map(ordem_atencao)
single_source = single_source.sort_values(
    ["_ord", "tag_abc", "desvio_mediano"],
    ascending=[True, True, False],
).drop(columns="_ord")

single_source = single_source[single_source["n_ops"].notna() & (single_source["n_ops"] > 0)]
print(f"📋 Pares Single-Source com OPs (fornecedor × produto): {len(single_source)}")
(single_source[display_cols]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .map(color_atencao, subset=["nivel_atencao"])
    .format({
        "lt_teorico": "{:.0f}d",
        "lt_realizado": "{:.0f}d",
        "desvio_mediano": "{:+.0f}d",
        "pct_no_prazo": "{:.1f}%",
    }, na_rep="—")
    .hide(axis="index")
)


📋 Pares Single-Source com OPs (fornecedor × produto): 35


product_name,fornecedor,fluxo,tag_abc,lt_teorico,lt_realizado,desvio_mediano,pct_no_prazo,n_ops,nivel_atencao
Undershirt Anti Suor Gola V Masculino,LUTESTIL,PA,A,90d,159d,+69d,0.0%,3.000000,🔴 Alto
Camisa FutureForm Masculino,MC & MC,PA,A,90d,150d,+60d,33.3%,3.000000,🔴 Alto
Vestido Chemise Sem Mangas FutureForm Feminino,PIXIE,PA,A,90d,131d,+41d,0.0%,2.000000,🔴 Alto
Vestido Curto Gola Canoa FutureForm Feminino,KABRIOLLI,PA,B,90d,139d,+49d,0.0%,3.000000,🔴 Alto
The Perfect Top Halter Feminino,RIZLLEP,PA,B,90d,130d,+40d,0.0%,2.000000,🔴 Alto
Spectrum Socks Low 2.0,MALHAS D'STEFANO,PA,B,60d,96d,+36d,66.7%,15.000000,🔴 Alto
Saia Midi Kyoto Feminino,MAURA,Tri,A,49d,138d,+29d,33.3%,9.000000,🟠 Moderado
Cueca Boxer Comfort Anti Suor Masculino,BAE BRASIL,PA,A,75d,102d,+27d,73.3%,15.000000,🟠 Moderado
Calça FutureForm Feminino,KABRIOLLI,PA,A,120d,143d,+23d,0.0%,3.000000,🟠 Moderado
Wingsuit Feminino,GOAT,Tri,A,40d,144d,+14d,18.2%,11.000000,🟠 Moderado


In [185]:
# Célula 10 — Nível 3: Top Spreads — Mesmo Produto + Faixa de Volume
# Parte A: Gráfico de barras horizontais (top 20 por spread bruto)
# Parte B: Tabela companion sem coluna oportunidade_ops_dias

# ====== PREPARAÇÃO DOS DADOS (compartilhada com heatmap abaixo) ======

spread_base = (
    df_ops.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order", "supplier_name"],
        observed=True,
    )
    .agg(
        lt_mediano = ("lead_time_realizado", "median"),
        n_ops      = ("op_code",             "count"),
    )
    .reset_index()
)
spread_base = spread_base[spread_base["n_ops"] >= CONFIG["min_ops_grafico"]]

contagem_forn = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["supplier_name"].nunique().reset_index(name="n_fornecedores")
)
spread_base = spread_base.merge(
    contagem_forn,
    on=["product_names", "volume_bucket", "is_finished_product_order"],
)
spread_base = spread_base[spread_base["n_fornecedores"] >= 2]

spread_min = (
    spread_base.sort_values("lt_mediano")
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_rapido",
        "supplier_name": "fornecedor_rapido",
        "n_ops":         "n_rapido",
    })
)

spread_max = (
    spread_base.sort_values("lt_mediano", ascending=False)
    .groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )
    .first().reset_index()
    [["product_names", "volume_bucket", "is_finished_product_order",
      "lt_mediano", "supplier_name", "n_ops"]]
    .rename(columns={
        "lt_mediano":    "lt_lento",
        "supplier_name": "fornecedor_lento",
        "n_ops":         "n_lento",
    })
)

n_total_grupo = (
    spread_base.groupby(
        ["product_names", "volume_bucket", "is_finished_product_order"], observed=True
    )["n_ops"].sum().reset_index(name="n_total_grupo")
)

spread_agg = (
    spread_min
    .merge(spread_max,    on=["product_names", "volume_bucket", "is_finished_product_order"])
    .merge(n_total_grupo, on=["product_names", "volume_bucket", "is_finished_product_order"])
)
spread_agg["spread_dias"] = spread_agg["lt_lento"] - spread_agg["lt_rapido"]
spread_agg["fluxo"]       = spread_agg["is_finished_product_order"].map({True: "PA", False: "Tri"})

# ====== PARTE A: GRÁFICO DE BARRAS (top 20 por spread bruto) ======

spread_top = spread_agg.sort_values("spread_dias", ascending=False).head(20).copy()

spread_top["label"] = (
    spread_top["product_names"].str[:28] + " | "
    + spread_top["volume_bucket"].astype(str) + " | "
    + spread_top["fluxo"]
)

fig = go.Figure(go.Bar(
    x=spread_top["spread_dias"],
    y=spread_top["label"],
    orientation="h",
    marker=dict(
        color=spread_top["spread_dias"],
        colorscale=[
            [0.0, COLORS["status_ok"]],
            [0.5, COLORS["status_atencao"]],
            [1.0, COLORS["status_critico"]],
        ],
        cmin=0, cmax=150,
        showscale=False,
    ),
    text=spread_top["spread_dias"].round(0).astype("Int64").astype(str) + "d",
    textposition="outside",
    customdata=spread_top[[
        "fornecedor_rapido", "lt_rapido",
        "fornecedor_lento",  "lt_lento",
        "n_total_grupo",
    ]].values,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Rápido: %{customdata[0]} (%{customdata[1]:.0f}d)<br>"
        "Lento:  %{customdata[2]} (%{customdata[3]:.0f}d)<br>"
        "n total no grupo: %{customdata[4]}<br>"
        "Spread: %{x:.0f}d<extra></extra>"
    ),
))
fig.update_layout(
    title="Top 20 Spreads — Oportunidade de Realocação (mesmo produto + faixa de volume)",
    template=TEMPLATE,
    xaxis_title="Spread de Lead Time (dias)",
    height=620,
    yaxis={"categoryorder": "total ascending"},
)
fig.show()

# ====== PARTE B: TABELA COMPANION (sem oportunidade_ops_dias) ======

display_cols_tab = [
    "product_names", "fluxo", "volume_bucket",
    "fornecedor_rapido", "lt_rapido", "n_rapido",
    "fornecedor_lento",  "lt_lento",  "n_lento",
    "spread_dias", "n_total_grupo",
]


def color_fluxo_cell(v):
    if v == "PA":
        return (f"background-color:{COLORS['fluxo_pa']};"
                "color:white; font-weight:600; text-align:center")
    if v == "Tri":
        return (f"background-color:{COLORS['fluxo_tri']};"
                "color:#374151; font-weight:600; text-align:center")
    return ""


(spread_top[display_cols_tab]
    .style
    .map(color_fluxo_cell, subset=["fluxo"])
    .background_gradient(subset=["spread_dias"], cmap="RdYlGn_r", vmin=0, vmax=150)
    .format({
        "lt_rapido":   "{:.0f}d",
        "lt_lento":    "{:.0f}d",
        "spread_dias": "{:.0f}d",
    })
    .hide(axis="index")
)


product_names,fluxo,volume_bucket,fornecedor_rapido,lt_rapido,n_rapido,fornecedor_lento,lt_lento,n_lento,spread_dias,n_total_grupo
The Perfect Top Feminino,PA,1000-1999,RDM,63d,5,BAE BRASIL,149d,11,86d,16
Spectrum Socks High 2.0,PA,1000-1999,BAE BRASIL,46d,4,MALHAS D'STEFANO,113d,6,67d,10
Daily Light T-shirt Masculino,PA,2000-4999,LUNELLI,74d,3,Lunelli Nordeste,119d,7,45d,10
Daily T-shirt Masculino,PA,2000-4999,BAE BRASIL,86d,3,DDAL,131d,5,45d,35
Tech T-shirt Gola U Masculino,PA,2000-4999,BAE BRASIL,96d,28,DDAL,127d,4,31d,62
"Core T-Shirt Masculino,Core T-shirt Masculino",PA,1000-1999,ABBA,106d,5,"Ges Confecção, Comercio e Serviços de Serigrafia LTDA",133d,4,27d,9
Daily T-shirt Feminino,PA,1000-1999,RIZLLEP,76d,4,ART LIVRE,103d,3,26d,7
"Core T-shirt Masculino,Core T-Shirt Masculino",Tri,1000-1999,FABIO,109d,4,NOVA FORMULA,123d,3,14d,7
Daily T-shirt Masculino,PA,1000-1999,ART LIVRE,98d,10,RIZLLEP,107d,11,10d,27
The Perfect Top Feminino,PA,2000-4999,RDM,102d,8,BAE BRASIL,106d,22,5d,30


In [186]:
# Célula 10.1 — Heatmap de Spread por Produto × Faixa de Volume (PA e Tri separados)
# Ordenação crescente: produtos com menor spread ficam no topo.
# Células sem amostra suficiente exibem "—" em cinza.


def montar_heatmap_spread(df_spread_agg, label_fluxo):
    sub = df_spread_agg[df_spread_agg["fluxo"] == label_fluxo].copy()
    if len(sub) == 0:
        return None, None

    sub["produto_label"] = sub["product_names"].str[:35]

    pivot_spread = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="spread_dias",
            aggfunc="median",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )
    pivot_n = (
        sub.pivot_table(
            index="produto_label",
            columns="volume_bucket",
            values="n_total_grupo",
            aggfunc="sum",
        )
        .reindex(columns=CONFIG["labels_volume"])
    )

    pivot_spread = pivot_spread.where(pivot_n >= CONFIG["min_ops_grafico"])

    ordem = pivot_spread.max(axis=1).sort_values(ascending=True).index
    return pivot_spread.loc[ordem], pivot_n.loc[ordem]


hm_tri, n_tri_hm = montar_heatmap_spread(spread_agg, "Tri")
hm_pa,  n_pa_hm  = montar_heatmap_spread(spread_agg, "PA")

subplots_data = [
    (hm, n_hm, lbl)
    for hm, n_hm, lbl in [(hm_tri, n_tri_hm, "Triangulação"), (hm_pa, n_pa_hm, "Produto Acabado")]
    if hm is not None and len(hm) > 0
]

n_cols = len(subplots_data)
fig_hm = make_subplots(
    rows=1, cols=n_cols,
    subplot_titles=[lbl for _, _, lbl in subplots_data],
    horizontal_spacing=0.18,
)

for idx, (hm, n_hm, lbl) in enumerate(subplots_data, start=1):
    texto = hm.copy().astype(object)
    for i in hm.index:
        for j in hm.columns:
            v = hm.loc[i, j]
            texto.loc[i, j] = f"{v:.0f}d" if pd.notna(v) else "—"

    fig_hm.add_trace(
        go.Heatmap(
            z=hm.values,
            x=list(hm.columns),
            y=list(hm.index),
            text=texto.values,
            texttemplate="%{text}",
            textfont=dict(size=10, color="#374151"),
            colorscale=[
                [0.0,  COLORS["status_ok"]],
                [0.33, "#FEF9C3"],
                [0.66, COLORS["status_atencao"]],
                [1.0,  COLORS["status_critico"]],
            ],
            zmid=50,
            zmin=0,
            zmax=150,
            showscale=(idx == n_cols),
            colorbar=dict(
                title="Spread (d)",
                x=1.02,
                tickvals=[0, 30, 60, 90, 120, 150],
                ticktext=["0d", "30d", "60d", "90d", "120d", "150d+"],
            ) if idx == n_cols else None,
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Volume: %{x}<br>"
                "Spread: %{z:.0f}d<extra></extra>"
            ),
        ),
        row=1, col=idx,
    )
    fig_hm.update_xaxes(title_text="Faixa de Volume", row=1, col=idx)

fig_hm.update_layout(
    title=(
        "Heatmap de Spread de Lead Time por Produto × Faixa de Volume<br>"
        "<sup>Ordenado por spread crescente — verde = menor spread</sup>"
    ),
    template=TEMPLATE,
    height=max(500, 22 * max(len(hm) for hm, _, _ in subplots_data) + 120),
)
fig_hm.show()


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_6708/270170204.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_6708/270170204.py:23: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_6708/270170204.py:14: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  sub.pivot_table(
/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_6708/270170204.py